In [14]:
import sys
from pathlib import Path

import os
import requests
from dotenv import load_dotenv

import warnings
# on va ignorer les messages avertissements a cause d'un RAGASS pas compatible mystral v2.0
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [15]:
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

load_dotenv()

TOKEN = os.getenv("API_KEY")
TOKEN_MISTRAL = os.getenv("MISTRAL_API_KEY")

API_URL_ASK = "http://127.0.0.1:8000/ask"
API_URL_ASK_DEBUG = "http://127.0.0.1:8000/ask/debug"


In [16]:
import requests
from typing import Any


def call_ask_api(question: str) -> dict[str, Any]:
    """
    Interroge l'endpoint /ask de l'API RAG.

    Parameters
    ----------
    question : str
        Question utilisateur à envoyer à l'API.

    Returns
    -------
    dict[str, Any]
        Réponse JSON de l'API.
    """
    resp = requests.post(
        API_URL_ASK,
        json={"question": question},
        headers={"x-api-key": TOKEN},
        timeout=60,
    )

    print("STATUS /ask :", resp.status_code)
    data = resp.json()
    print("ANSWER /ask :")
    print(data.get("answer", ""))

    return data


def run_rag_for_eval(question: str) -> dict[str, Any]:
    """
    Interroge l'endpoint /ask_debug pour récupérer la réponse
    ainsi que les contextes utilisés par le pipeline RAG.

    Parameters
    ----------
    question : str
        Question d'évaluation.

    Returns
    -------
    dict[str, Any]
        Dictionnaire contenant :
        - question
        - answer
        - contexts
        - error éventuel
    """
    resp = requests.post(
        API_URL_ASK_DEBUG,
        json={"question": question},
        headers={"x-api-key": TOKEN},
        timeout=60,
    )

    print(f"QUESTION : {question}")
    print("STATUS /ask_debug :", resp.status_code)

    try:
        data = resp.json()
    except Exception:
        data = {"raw_text": resp.text}

    if resp.status_code != 200:
        print("ERROR :", data)
        return {
            "question": question,
            "error": data,
            "answer": None,
            "contexts": None,
        }

    return {
        "question": question,
        "answer": data.get("answer"),
        "contexts": data.get("contexts"),
        "error": None,
    }


# ------------------------------------------------------------------
# Test rapide manuel de l'API
# ------------------------------------------------------------------
sample_question = "Quels événements culturels gratuits ont lieu à Montpellier ?"
sample_response = call_ask_api(sample_question)

STATUS /ask : 200
ANSWER /ask :
Je ne trouve pas d'événement correspondant, je suis un assistant qui ne peut vous conseiller que des événements culturels.


In [17]:
import requests
from typing import Any


def call_ask_api(question: str) -> dict[str, Any]:
    """
    Interroge l'endpoint /ask de l'API RAG.

    Parameters
    ----------
    question : str
        Question utilisateur à envoyer à l'API.

    Returns
    -------
    dict[str, Any]
        Réponse JSON de l'API.
    """
    resp = requests.post(
        API_URL_ASK,
        json={"question": question},
        headers={"x-api-key": TOKEN},
        timeout=60,
    )

    print("STATUS /ask :", resp.status_code)
    data = resp.json()
    print("ANSWER /ask :")
    print(data.get("answer", ""))

    return data


def run_rag_for_eval(question: str) -> dict[str, Any]:
    """
    Interroge l'endpoint /ask_debug pour récupérer la réponse
    ainsi que les contextes utilisés par le pipeline RAG.

    Parameters
    ----------
    question : str
        Question d'évaluation.

    Returns
    -------
    dict[str, Any]
        Dictionnaire contenant :
        - question
        - answer
        - contexts
        - error éventuel
    """
    resp = requests.post(
        API_URL_ASK_DEBUG,
        json={"question": question},
        headers={"x-api-key": TOKEN},
        timeout=60,
    )

    print(f"QUESTION : {question}")
    print("STATUS /ask_debug :", resp.status_code)

    try:
        data = resp.json()
    except Exception:
        data = {"raw_text": resp.text}

    if resp.status_code != 200:
        print("ERROR :", data)
        return {
            "question": question,
            "error": data,
            "answer": None,
            "contexts": None,
        }

    return {
        "question": question,
        "answer": data.get("answer"),
        "contexts": data.get("contexts"),
        "error": None,
    }


# ------------------------------------------------------------------
# Test rapide manuel de l'API
# ------------------------------------------------------------------
sample_question = "Quels événements culturels gratuits ont lieu à Montpellier ?"
sample_response = call_ask_api(sample_question)

STATUS /ask : 200
ANSWER /ask :
Je ne trouve pas d'événement correspondant, je suis un assistant qui ne peut vous conseiller que des événements culturels.


In [18]:
eval_questions_positive = [
    {
        "question": "Y a-t-il des expositions à Montpellier ?",
        "ground_truth": (
            "À Montpellier, les expositions présentes dans le corpus sont : "
            "L'association Miss'terre fait son expo d'été les 27 et 28 juin 2025 ; "
            "Exposition 'Montpellier, regarder la ville autrement. Archives et architecture' "
            "aux Archives de Montpellier le 20 septembre 2025 ; "
            "Exposition 'L'Europe et son patrimoine' à la Maison de l'Europe "
            "de Montpellier les 20 et 21 septembre 2025."
        ),
        "group": "positive",
    },
    {
        "question": "Quels événements ont lieu au Musée Fabre ?",
        "ground_truth": (
            "Au Musée Fabre à Montpellier, un événement présent dans le corpus est : "
            "la visite libre de la bibliothèque Jean Claparède les 20 et 21 septembre 2025, "
            "avec une lecture autour de l'architecture et de la poésie occitane."
        ),
        "group": "positive",
    },
    {
        "question": "Quels événements ont lieu à Montpellier en septembre 2025 ?",
        "ground_truth": (
            "À Montpellier en septembre 2025, les événements présents dans le corpus sont : "
            "la visite libre de la bibliothèque Jean Claparède au Musée Fabre les 20 et 21 septembre ; "
            "les conférences, expositions et dédicaces au Cercle Culturel Languedocien les 20 et 21 septembre ; "
            "l'exposition 'Montpellier, regarder la ville autrement' aux Archives de Montpellier le 20 septembre ; "
            "et l'exposition 'L'Europe et son patrimoine' les 20 et 21 septembre."
        ),
        "group": "positive",
    },
    {
        "question": "Quels événements ont lieu aux Archives de Montpellier ?",
        "ground_truth": (
            "Aux Archives de Montpellier, l'événement présent dans le corpus est : "
            "l'exposition 'Montpellier, regarder la ville autrement. Archives et architecture', "
            "le 20 septembre 2025."
        ),
        "group": "positive",
    },
    {
        "question": "Quels événements ont lieu le 20 septembre 2025 à Montpellier ?",
        "ground_truth": (
            "Le 20 septembre 2025 à Montpellier, les événements présents dans le corpus sont : "
            "la visite libre de la bibliothèque Jean Claparède au Musée Fabre ; "
            "les conférences, expositions et dédicaces au Cercle Culturel Languedocien ; "
            "et l'exposition 'Montpellier, regarder la ville autrement' aux Archives de Montpellier."
        ),
        "group": "positive",
    },
    {
        "question": "Quels événements ont lieu le week-end du 20 au 21 septembre 2025 à Montpellier ?",
        "ground_truth": (
            "Le week-end du 20 et 21 septembre 2025 à Montpellier comprend : "
            "la visite libre de la bibliothèque Jean Claparède au Musée Fabre ; "
            "les conférences, expositions et dédicaces au Cercle Culturel Languedocien ; "
            "et l'exposition 'L'Europe et son patrimoine'."
        ),
        "group": "positive",
    },
]

eval_questions_ambiguous = [
    {
        "question": "Quels événements culturels sont proposés à Montpellier ?",
        "ground_truth": (
            "Plusieurs événements culturels sont proposés à Montpellier dans le corpus : "
            "la visite libre de la bibliothèque Jean Claparède au Musée Fabre ; "
            "des conférences, expositions et dédicaces au Cercle Culturel Languedocien ; "
            "l'exposition 'Montpellier, regarder la ville autrement' aux Archives ; "
            "et l'exposition 'L'Europe et son patrimoine'."
        ),
        "group": "ambiguous",
    },
    {
        "question": "Quels événements gratuits ont lieu à Montpellier ?",
        "ground_truth": (
            "Le corpus ne permet pas de confirmer explicitement la gratuité des événements, "
            "aucun événement ne peut donc être identifié avec certitude comme gratuit."
        ),
        "group": "ambiguous",
    },
]

eval_questions_negative = [
    {
        "question": "Y a-t-il des concerts de rock à Montpellier?",
        "ground_truth": (
            "Je ne trouve pas d'événement correspondant, je suis un assistant qui "
            "ne peut vous conseiller que des événements culturels."
        ),
        "group": "negative",
    },
    {
        "question": "Quels événements ont lieu à Paris ?",
        "ground_truth": (
            "Je ne trouve pas d'événement correspondant, je suis un assistant qui "
            "ne peut vous conseiller que des événements culturels."
        ),
        "group": "negative",
    },
]

eval_questions = (
    eval_questions_positive
    + eval_questions_ambiguous
    + eval_questions_negative
)

len(eval_questions)

10

In [19]:
#construction dataset evaluation
rows = []

for item in eval_questions:
    result = run_rag_for_eval(item["question"])

    rows.append(
        {
            "question": item["question"],
            "answer": result["answer"],
            "contexts": result["contexts"],
            "ground_truth": item["ground_truth"],
            "group": item["group"],
            "error": result["error"],
        }
    )

rows[:1]

QUESTION : Y a-t-il des expositions à Montpellier ?
STATUS /ask_debug : 200
QUESTION : Quels événements ont lieu au Musée Fabre ?
STATUS /ask_debug : 200
QUESTION : Quels événements ont lieu à Montpellier en septembre 2025 ?
STATUS /ask_debug : 200
QUESTION : Quels événements ont lieu aux Archives de Montpellier ?
STATUS /ask_debug : 200
QUESTION : Quels événements ont lieu le 20 septembre 2025 à Montpellier ?
STATUS /ask_debug : 200
QUESTION : Quels événements ont lieu le week-end du 20 au 21 septembre 2025 à Montpellier ?
STATUS /ask_debug : 200
QUESTION : Quels événements culturels sont proposés à Montpellier ?
STATUS /ask_debug : 200
QUESTION : Quels événements gratuits ont lieu à Montpellier ?
STATUS /ask_debug : 200
QUESTION : Y a-t-il des concerts de rock à Montpellier?
STATUS /ask_debug : 200
QUESTION : Quels événements ont lieu à Paris ?
STATUS /ask_debug : 200


[{'question': 'Y a-t-il des expositions à Montpellier ?',
  'answer': 'Titre : L\'association Miss\'terre fait son expo d\'été !\n\nLieu : Association Miss\'terre, Montpellier\nDate : 27-28 juin 2025\nDescription : Exposition organisée par l\'association Miss\'terre dans le cadre de son événement estival.\n\nPourquoi cet événement pourrait vous intéresser : Il s\'agit d\'une exposition à Montpellier, même si la date est en juin 2025 (et non en mars 2026 comme la date actuelle). Cela pourrait correspondre à vos attentes si vous cherchez des expositions dans cette ville.\n\n---\n\nTitre : Exposition "Montpellier, regarder la ville autrement. Archives et architecture"\n\nLieu : Archives de Montpellier\nDate : 20 septembre 2025\nDescription : Exposition présentant des documents issus des fonds des Archives de Montpellier sur l\'architecture de la ville, à travers cinq monuments emblématiques.\n\nPourquoi cet événement pourrait vous intéresser : Cette exposition met en valeur le patrimoine 

In [20]:
import pandas as pd

df_eval_raw = pd.DataFrame(rows)

print(df_eval_raw.shape)
df_eval_raw[["question", "group", "error"]]

(10, 6)


,question,group,error
0,Y a-t-il des expositions à Montpellier ?,positive,None
1,Quels événements ont lieu au Musée Fabre ?,positive,None
2,Quels événements ont lieu à Montpellier en sep...,positive,None
3,Quels événements ont lieu aux Archives de Mont...,positive,None
4,Quels événements ont lieu le 20 septembre 2025...,positive,None
5,Quels événements ont lieu le week-end du 20 au...,positive,None
6,Quels événements culturels sont proposés à Mon...,ambiguous,None
7,Quels événements gratuits ont lieu à Montpelli...,ambiguous,None
8,Y a-t-il des concerts de rock à Montpellier?,negative,None
9,Quels événements ont lieu à Paris ?,negative,None


In [21]:
for i, row in enumerate(rows):
    assert isinstance(row["question"], str), f"question row {i} invalide"
    assert isinstance(row["ground_truth"], str), f"ground_truth row {i} invalide"

    if row["answer"] is not None:
        assert isinstance(row["answer"], str), f"answer row {i} invalide"

    if row["contexts"] is not None:
        assert isinstance(row["contexts"], list), f"contexts row {i} invalide"
        assert all(isinstance(c, str) for c in row["contexts"]), (
            f"contexts row {i} contient autre chose que str"
        )

print("Format OK pour les rows")

Format OK pour les rows


In [22]:
from ragas.dataset_schema import EvaluationDataset

ragas_rows = [
    {
        "user_input": row["question"],
        "response": row["answer"],
        "retrieved_contexts": row["contexts"],
        "reference": row["ground_truth"],
        "group": row["group"],
    }
    for row in rows
    if row["answer"] is not None and row["contexts"] is not None
]

ragas_dataset = EvaluationDataset.from_list(ragas_rows)

print(ragas_rows[0])
print(ragas_dataset)

{'user_input': 'Y a-t-il des expositions à Montpellier ?', 'response': 'Titre : L\'association Miss\'terre fait son expo d\'été !\n\nLieu : Association Miss\'terre, Montpellier\nDate : 27-28 juin 2025\nDescription : Exposition organisée par l\'association Miss\'terre dans le cadre de son événement estival.\n\nPourquoi cet événement pourrait vous intéresser : Il s\'agit d\'une exposition à Montpellier, même si la date est en juin 2025 (et non en mars 2026 comme la date actuelle). Cela pourrait correspondre à vos attentes si vous cherchez des expositions dans cette ville.\n\n---\n\nTitre : Exposition "Montpellier, regarder la ville autrement. Archives et architecture"\n\nLieu : Archives de Montpellier\nDate : 20 septembre 2025\nDescription : Exposition présentant des documents issus des fonds des Archives de Montpellier sur l\'architecture de la ville, à travers cinq monuments emblématiques.\n\nPourquoi cet événement pourrait vous intéresser : Cette exposition met en valeur le patrimoine

In [23]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_mistralai import ChatMistralAI

from ragas import evaluate
from ragas.dataset_schema import EvaluationDataset
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    answer_relevancy,
    context_precision,
    context_recall,
    faithfulness,
)

llm = ChatMistralAI(
    model="mistral-small-latest",
    api_key=TOKEN_MISTRAL.strip(),
    temperature=0,
)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

evaluator_llm = LangchainLLMWrapper(llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)

small_ragas_dataset = EvaluationDataset.from_list(ragas_rows[:10])

result = evaluate(
    dataset=small_ragas_dataset,
    metrics=[
        faithfulness,
        context_precision,
        context_recall,
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    raise_exceptions=False,
)

print(result)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]

Exception raised in Job[23]: HTTPStatusError(Error response 502 while fetching https://api.mistral.ai/v1/chat/completions: {
  "message":"An invalid response was received from the upstream server",
  "request_id":"ad22f22867ab860eec5f527afa191f33"
})
Exception raised in Job[27]: HTTPStatusError(Error response 502 while fetching https://api.mistral.ai/v1/chat/completions: {
  "message":"An invalid response was received from the upstream server",
  "request_id":"58a48e4bd5c29423235dba1181b4bf64"
})
Exception raised in Job[25]: HTTPStatusError(Error response 502 while fetching https://api.mistral.ai/v1/chat/completions: {
  "message":"An invalid response was received from the upstream server",
  "request_id":"1e9c39104e99e202611aae2d42e8747b"
})
Exception raised in Job[22]: HTTPStatusError(Error response 502 while fetching https://api.mistral.ai/v1/chat/completions: {
  "message":"An invalid response was received from the upstream server",
  "request_id":"e71d995f4959dd7fb2da051421a13037"

{'faithfulness': 0.6734, 'context_precision': 0.5476, 'context_recall': 0.6296}


In [25]:

result_df = result.to_pandas()
result_df

,user_input,retrieved_contexts,response,reference,faithfulness,context_precision,context_recall
0,Y a-t-il des expositions à Montpellier ?,[L'association Miss'terre fait son expo d'été ...,Titre : L'association Miss'terre fait son expo...,"À Montpellier, les expositions présentes dans ...",0.761905,1.000000,1.000000
1,Quels événements ont lieu au Musée Fabre ?,"[Visite commentée : « Un musée, des bâtiments,...","Titre : Visite commentée : « Un musée, des bât...","Au Musée Fabre à Montpellier, un événement pré...",1.000000,0.333333,0.000000
2,Quels événements ont lieu à Montpellier en sep...,[Visite guidée du cimetière protestant de Mont...,Titre : Visite guidée du cimetière protestant ...,"À Montpellier en septembre 2025, les événement...",1.000000,NaN,0.500000
3,Quels événements ont lieu aux Archives de Mont...,"[Exposition ""Montpellier, regarder la ville au...",Voici les événements correspondant à votre dem...,"Aux Archives de Montpellier, l'événement prése...",1.000000,1.000000,1.000000
4,Quels événements ont lieu le 20 septembre 2025...,[Visite guidée du cimetière protestant de Mont...,Titre : Visite guidée du cimetière protestant ...,"Le 20 septembre 2025 à Montpellier, les événem...",1.000000,0.000000,0.333333
5,Quels événements ont lieu le week-end du 20 au...,[Visite libre de la bibliothèque Jean Claparèd...,Titre : Visite libre de la bibliothèque Jean C...,Le week-end du 20 et 21 septembre 2025 à Montp...,0.625000,0.000000,0.333333
6,Quels événements culturels sont proposés à Mon...,"[Exposition ""Montpellier, regarder la ville au...",Voici les événements culturels proposés à Mont...,Plusieurs événements culturels sont proposés à...,NaN,0.500000,0.500000
7,Quels événements gratuits ont lieu à Montpelli...,[Visite libre de la bibliothèque Jean Claparèd...,"Je ne trouve pas d'événement correspondant, je...",Le corpus ne permet pas de confirmer explicite...,0.000000,NaN,NaN
8,Y a-t-il des concerts de rock à Montpellier?,[Concert « Nocturnàlia »\n💃 Hommage à Max Rouq...,"Je ne trouve pas d'événement correspondant, je...","Je ne trouve pas d'événement correspondant, je...",0.000000,NaN,1.000000
9,Quels événements ont lieu à Paris ?,[La Dame de Pierre\nDécouvrez le récit légenda...,"Je ne trouve pas d'événement correspondant, je...","Je ne trouve pas d'événement correspondant, je...",NaN,1.000000,1.000000


In [26]:
print(result)

{'faithfulness': 0.6734, 'context_precision': 0.5476, 'context_recall': 0.6296}
